# Unit 4.1 Exercise

In [4]:
import re
import math
from collections import Counter, defaultdict


data = [
    ("Free money now!!!", "SPAM"),
    ("Hi mom, how are you?", "HAM"),
    ("Lowest price for your meds", "SPAM"),
    ("Are we still on for dinner?", "HAM"),
    ("Win a free iPhone today", "SPAM"),
    ("Let's catch up tomorrow at the office", "HAM"),
    ("Meeting at 3 PM tomorrow", "HAM"),
    ("Get 50% off, limited time!", "SPAM"),
    ("Team meeting in the office", "HAM"),
    ("Click here for prizes!", "SPAM"),
    ("Can you send the report?", "HAM"),
]

test_sentences = [
    "Limited offer, click here!",
    "Meeting at 2 PM with the manager."
]


def tokenize(text: str):
    # Simple tokenizer:
    # - lowercases
    # - keeps alphanumeric words (drops punctuation)
    # - splits into tokens
    text = text.lower()
    return re.findall(r"[a-z0-9]+", text)


for s in test_sentences:
    print(s, "->", tokenize(s))


Limited offer, click here! -> ['limited', 'offer', 'click', 'here']
Meeting at 2 PM with the manager. -> ['meeting', 'at', '2', 'pm', 'with', 'the', 'manager']


### a. Generate a Bag of Words (Word Frequency)

In [5]:
docs_by_class = defaultdict(list)
for text, label in data:
    docs_by_class[label].append(tokenize(text))

vocab = sorted({tok for label in docs_by_class for doc in docs_by_class[label] for tok in doc})
V = len(vocab)

token_counts = {}
total_tokens = {}
for label, docs in docs_by_class.items():
    c = Counter()
    for doc in docs:
        c.update(doc)
    token_counts[label] = c
    total_tokens[label] = sum(c.values())

print("Vocabulary size |V| =", V)

Vocabulary size |V| = 45


### b. Calculate the Prior for HAM and SPAM

In [6]:
num_docs = len(data)
class_doc_counts = {label: len(docs) for label, docs in docs_by_class.items()}
priors = {label: class_doc_counts[label] / num_docs for label in class_doc_counts}

print("Priors:")

Priors:


### c. Calculate the Likelihood P(token | class)

In [7]:
likelihoods = {}

for label in token_counts:
    denom = total_tokens[label] + V
    likelihoods[label] = {}
    for tok in vocab:
        likelihoods[label][tok] = (token_counts[label][tok] + 1) / denom


### d. Classify the Test Sentences

In [8]:
def predict_manual(text: str):
    tokens = tokenize(text)
    tf = Counter(tokens)

    scores = {}
    for label in priors:
        score = math.log(priors[label])
        for tok, count in tf.items():
            if tok in likelihoods[label]:
                score += count * math.log(likelihoods[label][tok])
        scores[label] = score

    pred = max(scores, key=scores.get)
    return pred, scores, tokens


# Outside the function (no indentation)
for s in test_sentences:
    pred, scores, toks = predict_manual(s)
    print(f"\nSentence: {s}")
    print("Tokens:", toks)
    for label, sc in scores.items():
        print(f"  log P({label} | d) ∝ {sc:.6f}")
    print("=> Predicted:", pred)



Sentence: Limited offer, click here!
Tokens: ['limited', 'offer', 'click', 'here']
  log P(SPAM | d) ∝ -11.323094
  log P(HAM | d) ∝ -13.714479
=> Predicted: SPAM

Sentence: Meeting at 2 PM with the manager.
Tokens: ['meeting', 'at', '2', 'pm', 'with', 'the', 'manager']
  log P(SPAM | d) ∝ -17.607228
  log P(HAM | d) ∝ -13.807261
=> Predicted: HAM


## Using Scikit-Learn

In [9]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

X_text = [t for t, y in data]
y = [y for t, y in data]

vectorizer = CountVectorizer(lowercase=True, token_pattern=r"[A-Za-z0-9]+")
X = vectorizer.fit_transform(X_text)

clf = MultinomialNB(alpha=1.0)
clf.fit(X, y)

X_test = vectorizer.transform(test_sentences)
preds = clf.predict(X_test)
probas = clf.predict_proba(X_test)
classes = clf.classes_

for s, pred, pr in zip(test_sentences, preds, probas):
    print(f"\nSentence: {s}")
    print("Predicted:", pred)
    for cls, prob in zip(classes, pr):
        print(f"  P({cls} | d) = {prob:.6f}")



Sentence: Limited offer, click here!
Predicted: SPAM
  P(HAM | d) = 0.083832
  P(SPAM | d) = 0.916168

Sentence: Meeting at 2 PM with the manager.
Predicted: HAM
  P(HAM | d) = 0.978118
  P(SPAM | d) = 0.021882
